In [10]:
import pandas as pd
import numpy as np
import pickle

# Load behaviors
behaviors = pd.read_pickle("processed_data/behaviors_clean.pkl")

# Load news vectors
with open("processed_data/news_vector_dict.pkl", "rb") as f:
    news_vector_dict = pickle.load(f)

# Load time features
time_features = np.load("processed_data/time_features.npy")

print("Behaviors:", behaviors.shape)
print("Time features:", time_features.shape)

Behaviors: (156965, 5)
Time features: (156965, 5)


In [11]:
sample_vector = next(iter(news_vector_dict.values()))
news_dim = len(sample_vector)

print("News vector dim:", news_dim)

News vector dim: 868


In [12]:
def get_history_embedding(history):
    vectors = []
    
    for news_id in history:
        if news_id in news_vector_dict:
            vectors.append(news_vector_dict[news_id])
    
    if len(vectors) == 0:
        return np.zeros(news_dim)
    
    return np.mean(vectors, axis=0)

In [13]:
from tqdm import tqdm

user_vectors = []

for idx, row in tqdm(behaviors.iterrows(), total=len(behaviors)):
    history = row["history"]
    
    history_emb = get_history_embedding(history)
    time_emb = time_features[idx]
    
    user_vec = np.concatenate([history_emb, time_emb])
    user_vectors.append(user_vec)

user_vectors = np.array(user_vectors)

print("User vector shape:", user_vectors.shape)

100%|██████████| 156965/156965 [00:07<00:00, 21985.74it/s]


User vector shape: (156965, 873)


In [14]:
from sklearn.preprocessing import normalize

user_vectors = normalize(user_vectors)

In [15]:
np.save("processed_data/user_vectors.npy", user_vectors)

In [16]:
print(user_vectors.shape)
print(user_vectors[0][:10])

(156965, 873)
[-0.00972811 -0.00689524  0.00457873  0.00701733  0.01072084  0.00089605
  0.00349178 -0.01243074  0.01839877 -0.00113132]


# Build Final Interaction Features

In [8]:
# Load data
behaviors = pd.read_pickle("processed_data/behaviors_clean.pkl")
user_vectors = np.load("processed_data/user_vectors.npy")

with open("processed_data/news_vector_dict.pkl", "rb") as f:
    news_vector_dict = pickle.load(f)

In [9]:
user_dim = user_vectors.shape[1]
news_dim = len(next(iter(news_vector_dict.values())))

print("User dim:", user_dim)
print("News dim:", news_dim)

User dim: 1641
News dim: 1636


In [6]:
X = []
y = []

for idx, row in behaviors.iterrows():
    user_vec = user_vectors[idx]
    
    for news_id, label in row["impressions"]:
        if news_id not in news_vector_dict:
            continue
        
        news_vec = news_vector_dict[news_id]
        
        feature_vec = np.concatenate([user_vec, news_vec])
        
        X.append(feature_vec)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Final dataset shape:", X.shape)
print("Labels shape:", y.shape)



: 